In [ ]:
import polars as pl
import polars.selectors as cs
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import warnings

SyntaxError: invalid decimal literal (2027610727.py, line 8)

In [8]:
train_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/train_eda.parquet")
test_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/test_eda.parquet")

In [9]:
train_df = train_df.with_columns([
    (pl.col("amt_credit") / pl.col("amt_income_total")).alias("credit_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_income_total")).alias("annuity_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_credit")).alias("annuity_to_credit_ratio"),
    (pl.col("days_employed") / pl.col("days_birth")).alias("employed_to_age_ratio"),
])

In [10]:
print(train_df.shape, test_df.shape)

(307511, 127) (48744, 121)


In [11]:
for col in ["ext_source_1", "ext_source_2", "ext_source_3"]:
    median_val = train_df[col].median()
    train_df = train_df.with_columns([
        pl.col(col).is_null().alias(f"{col}_was_missing"),
        pl.col(col).fill_null(median_val).alias(col),
    ])

In [12]:
train_df = train_df.with_columns([
    (pl.col("amt_credit") / pl.col("amt_income_total")).alias("credit_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_income_total")).alias("annuity_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_credit")).alias("annuity_to_credit_ratio"),
    (pl.col("amt_goods_price") / pl.col("amt_credit")).alias("goods_price_to_credit_ratio"),
])

test_df = test_df.with_columns([
    (pl.col("amt_credit") / pl.col("amt_income_total")).alias("credit_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_income_total")).alias("annuity_to_income_ratio"),
    (pl.col("amt_annuity") / pl.col("amt_credit")).alias("annuity_to_credit_ratio"),
    (pl.col("amt_goods_price") / pl.col("amt_credit")).alias("goods_price_to_credit_ratio"),
])

In [13]:
for col in ["credit_to_income_ratio", "annuity_to_income_ratio", "annuity_to_credit_ratio", "goods_price_to_credit_ratio"]:
    n_inf = train_df.filter(pl.col(col).is_infinite() | pl.col(col).is_nan()).height
    print(f"{col}: {n_inf} проблемных строк")

credit_to_income_ratio: 0 проблемных строк
annuity_to_income_ratio: 0 проблемных строк
annuity_to_credit_ratio: 0 проблемных строк
goods_price_to_credit_ratio: 0 проблемных строк


In [14]:
categorical_features = train_df.select(pl.col(pl.Utf8)).columns

cardinalities = {c: train_df[c].n_unique() for c in categorical_features}
low_card = [c for c, n in cardinalities.items() if n <= 10]
high_card = [c for c, n in cardinalities.items() if n > 10]

print("One-Hot (низкая cardinality):", low_card)
print("Target Encoding (высокая cardinality):", high_card)

One-Hot (низкая cardinality): ['name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'name_type_suite', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'weekday_appr_process_start', 'fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']
Target Encoding (высокая cardinality): ['occupation_type', 'organization_type']


In [15]:
train_df = train_df.to_dummies(columns=low_card)
test_df = test_df.to_dummies(columns=low_card)

In [16]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
print("Есть в train, нет в test:", train_cols - test_cols)
print("Есть в test, нет в train:", test_cols - train_cols)

Есть в train, нет в test: {'ext_source_1_was_missing', 'name_family_status_Unknown', 'ext_source_3_was_missing', 'employed_to_age_ratio', 'name_income_type_Maternity leave', 'target', 'days_employed_anomaly', 'ext_source_2_was_missing', 'code_gender_XNA'}
Есть в test, нет в train: set()


In [17]:
# was_missing флаги для EXT_SOURCE — важно: медиану берём из train_df (уже посчитана), 
# просто применяем к test_df
for col in ["ext_source_1", "ext_source_2", "ext_source_3"]:
    median_val = train_df[col].median()  # медиана ИЗ train, не пересчитывать на test!
    test_df = test_df.with_columns([
        pl.col(col).is_null().alias(f"{col}_was_missing"),
        pl.col(col).fill_null(median_val).alias(col),
    ])

# days_employed_anomaly
test_df = test_df.with_columns(
    (pl.col("days_employed") == 365243).alias("days_employed_anomaly")
)

# employed_to_age_ratio
test_df = test_df.with_columns(
    (pl.col("days_employed") / pl.col("days_birth")).alias("employed_to_age_ratio")
)

In [18]:
missing_in_test = {'name_family_status_Unknown', 'name_income_type_Maternity leave', 'code_gender_XNA'}

test_df = test_df.with_columns([
    pl.lit(0).alias(col) for col in missing_in_test
])

In [19]:
train_cols = set(train_df.columns) - {"target"}
test_cols = set(test_df.columns)
print("Расхождение:", train_cols - test_cols, "|", test_cols - train_cols)

Расхождение: set() | set()


In [20]:
from sklearn.model_selection import StratifiedKFold

def target_encode_cv(df, col, target_col="target", n_splits=5, smoothing=10):
    global_mean = df[target_col].mean()
    encoded = np.zeros(df.height)
    y = df[target_col].to_numpy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in skf.split(df, y):
        fold_train = df[train_idx]
        stats = fold_train.group_by(col).agg([
            pl.col(target_col).mean().alias("cat_mean"),
            pl.count().alias("cat_count"),
        ])
        # сглаживание: чем меньше примеров категории, тем ближе к global_mean
        stats = stats.with_columns(
            ((pl.col("cat_mean") * pl.col("cat_count") + global_mean * smoothing) 
             / (pl.col("cat_count") + smoothing)).alias("smoothed_mean")
        )
        val_df = df[val_idx].join(stats.select([col, "smoothed_mean"]), on=col, how="left")
        encoded[val_idx] = val_df["smoothed_mean"].fill_null(global_mean).to_numpy()

    return encoded

In [21]:
for col in high_card:
    train_df = train_df.with_columns(pl.Series(f"{col}_te", target_encode_cv(train_df, col)))

C:\Users\kozin\AppData\Local\Temp\ipykernel_10424\2051515399.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("cat_count"),
C:\Users\kozin\AppData\Local\Temp\ipykernel_10424\2051515399.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("cat_count"),


In [22]:
for col in high_card:
    global_mean = train_df["target"].mean()
    stats = train_df.group_by(col).agg([
        pl.col("target").mean().alias("cat_mean"),
        pl.count().alias("cat_count"),
    ]).with_columns(
        ((pl.col("cat_mean") * pl.col("cat_count") + global_mean * 10) 
         / (pl.col("cat_count") + 10)).alias("smoothed_mean")
    )
    test_df = test_df.join(stats.select([col, "smoothed_mean"]), on=col, how="left") \
        .rename({"smoothed_mean": f"{col}_te"}) \
        .with_columns(pl.col(f"{col}_te").fill_null(global_mean))

C:\Users\kozin\AppData\Local\Temp\ipykernel_10424\3097615825.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("cat_count"),


In [23]:
train_df = train_df.drop(high_card)
test_df = test_df.drop(high_card)

In [24]:
te_cols = [c for c in train_df.columns if c.endswith("_te")]
print(train_df.select(te_cols).describe())

shape: (9, 3)
┌────────────┬────────────────────┬──────────────────────┐
│ statistic  ┆ occupation_type_te ┆ organization_type_te │
│ ---        ┆ ---                ┆ ---                  │
│ str        ┆ f64                ┆ f64                  │
╞════════════╪════════════════════╪══════════════════════╡
│ count      ┆ 307511.0           ┆ 307511.0             │
│ null_count ┆ 0.0                ┆ 0.0                  │
│ mean       ┆ 0.085615           ┆ 0.080733             │
│ std        ┆ 0.019393           ┆ 0.019717             │
│ min        ┆ 0.046827           ┆ 0.028239             │
│ 25%        ┆ 0.06661            ┆ 0.061223             │
│ 50%        ┆ 0.080729           ┆ 0.085439             │
│ 75%        ┆ 0.105452           ┆ 0.093482             │
│ max        ┆ 0.173876           ┆ 0.16025              │
└────────────┴────────────────────┴──────────────────────┘


In [25]:
bureau_df = pl.read_csv("C:/CreditScoringMLE/ml_service/app/data/bureau.csv")
bureau_df = bureau_df.select(pl.all().name.to_lowercase())

bureau_agg = bureau_df.group_by("sk_id_curr").agg([
    pl.len().alias("bureau_credits_cnt"),
    (pl.col("credit_active") == "Active").sum().alias("bureau_active_cnt"),
    pl.col("days_credit").max().alias("bureau_days_credit_max"),
    pl.col("days_credit").mean().alias("bureau_days_credit_mean"),
    pl.col("amt_credit_sum").sum().alias("bureau_credit_sum_total"),
    pl.col("amt_credit_sum_debt").sum().alias("bureau_credit_debt_total"),
    pl.col("amt_credit_sum_overdue").sum().alias("bureau_overdue_total"),
])

train_df = train_df.join(bureau_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")
test_df = test_df.join(bureau_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")

In [ ]:
bureau_cols = ["bureau_credits_cnt", "bureau_active_cnt", "bureau_credit_sum_total", 
               "bureau_credit_debt_total", "bureau_overdue_total"]
train_df = train_df.with_columns([pl.col(c).fill_null(0) for c in bureau_cols])
test_df = test_df.with_columns([pl.col(c).fill_null(0) for c in bureau_cols])



In [27]:
print(train_df.select(bureau_cols + ["bureau_days_credit_max"]).describe())

shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ bureau_credi ┆ bureau_activ ┆ bureau_cred ┆ bureau_cred ┆ bureau_over ┆ bureau_days │
│ ---        ┆ ts_cnt       ┆ e_cnt        ┆ it_sum_tota ┆ it_debt_tot ┆ due_total   ┆ _credit_max │
│ str        ┆ ---          ┆ ---          ┆ l           ┆ al          ┆ ---         ┆ ---         │
│            ┆ f64          ┆ f64          ┆ ---         ┆ ---         ┆ f64         ┆ f64         │
│            ┆              ┆              ┆ f64         ┆ f64         ┆             ┆             │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 307511.0     ┆ 307511.0     ┆ 307511.0    ┆ 307511.0    ┆ 307511.0    ┆ 263491.0    │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 44020.0     │
│ mean       ┆ 4.765114     ┆ 1.762275     ┆ 1.6758e6    ┆ 548941.5883 ┆ 191.

In [28]:
median_days_credit = train_df["bureau_days_credit_max"].median()
train_df = train_df.with_columns(pl.col("bureau_days_credit_max").fill_null(median_days_credit))
test_df = test_df.with_columns(pl.col("bureau_days_credit_max").fill_null(median_days_credit))

In [29]:
prev_app_df = pl.read_csv("C:/CreditScoringMLE/ml_service/app/data/previous_application.csv")
prev_app_df = prev_app_df.select(pl.all().name.to_lowercase())

prev_agg = prev_app_df.group_by("sk_id_curr").agg([
    pl.len().alias("prev_app_cnt"),
    (pl.col("name_contract_status") == "Approved").sum().alias("prev_app_approved_cnt"),
    (pl.col("name_contract_status") == "Refused").sum().alias("prev_app_refused_cnt"),
    pl.col("amt_credit").mean().alias("prev_app_credit_mean"),
    pl.col("amt_annuity").mean().alias("prev_app_annuity_mean"),
    pl.col("days_decision").mean().alias("prev_app_days_decision_mean"),
])

train_df = train_df.join(prev_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")
test_df = test_df.join(prev_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")

count_cols = ["prev_app_cnt", "prev_app_approved_cnt", "prev_app_refused_cnt"]
train_df = train_df.with_columns([pl.col(c).fill_null(0) for c in count_cols])
test_df = test_df.with_columns([pl.col(c).fill_null(0) for c in count_cols])
# mean-колонки (credit_mean, annuity_mean, days_decision_mean) — заполни медианой из train, как с bureau

In [30]:
train_df.write_parquet("C:/CreditScoringMLE/ml_service/app/data/train_features.parquet")
test_df.write_parquet("C:/CreditScoringMLE/ml_service/app/data/test_features.parquet")